# DyTopo — Token Consumption & Cost Analysis

Load saved results from **any completed run** (Hyperspace or Vast.ai) and produce:
- Per-dataset token breakdown (prompt vs completion)
- Estimated API cost per model/pricing
- Comparison to paper Table 8 (DyTopo: 9,453 tok/problem on HumanEval)
- Topology edge density evolution across rounds
- Full results summary vs paper Table 2

## How to use
1. Set `RESULTS_DIR` to the folder containing `*_results.json` files
2. Set `LLM_MODEL` and `BACKEND` to match what generated those results
3. Run all cells

In [ ]:
# ── Cell 1: Config — point at your results folder ─────────────────────────────
from pathlib import Path

# Change to match your run:
RESULTS_DIR = Path('./results_hyperspace')     # or /workspace/results_mimo etc.
LLM_MODEL   = "claude-haiku-4-5-20251001"      # model that produced the results
BACKEND     = "hyperspace"                     # "hyperspace" or "vastai"
MODEL_LABEL = f"{BACKEND} / {LLM_MODEL}"

print(f"Loading results from : {RESULTS_DIR}")
print(f"Model                : {LLM_MODEL}")
available = list(RESULTS_DIR.glob('*_results.json'))
print(f"Result files found   : {[f.name for f in available]}")

In [ ]:
print('Loading datasets...')
from datasets import load_dataset
import random, json, re, subprocess, sys
random.seed(42)

humaneval_ds = load_dataset('openai/openai_humaneval', split='test')
math500_ds   = load_dataset('HuggingFaceH4/MATH-500',  split='test')
omnimath_ds  = load_dataset('KbsdJames/Omni-MATH',     split='test')
print(f'HumanEval {len(humaneval_ds)} | MATH-500 {len(math500_ds)} | Omni-MATH {len(omnimath_ds)}')

humaneval_problems = [
    {'id': x['task_id'], 'problem': x['prompt'], 'canonical': x['canonical_solution'],
     'test': x['test'], 'entry_point': x['entry_point'], 'dataset': 'humaneval'}
    for x in humaneval_ds
]
math500_problems = [
    {'id': f'math500_{i}', 'problem': x['problem'], 'answer': x['answer'],
     'subject': x['subject'], 'level': x['level'], 'dataset': 'math500'}
    for i, x in enumerate(math500_ds)
]

def sample_omnimath(ds, n_domains=7, n_per_domain=10, seed=42):
    rng = random.Random(seed)
    dom_key = 'domain' if 'domain' in ds.column_names else 'subject'
    by_dom = {}
    for i, x in enumerate(ds):
        raw = x.get(dom_key, 'unknown')
        d = (raw[0] if isinstance(raw, list) and raw else str(raw)).split('/')[0].strip()
        by_dom.setdefault(d, []).append((i, x))
    top = sorted(by_dom, key=lambda k: -len(by_dom[k]))[:n_domains]
    out = []
    for dom in top:
        for idx, x in rng.sample(by_dom[dom], min(n_per_domain, len(by_dom[dom]))):
            out.append({'id': f'omnimath_{idx}',
                        'problem': x.get('problem', x.get('question', '')),
                        'answer': x.get('answer', x.get('solution', '')),
                        'domain': dom, 'dataset': 'omnimath'})
    return out

omnimath_problems = sample_omnimath(omnimath_ds)
apps_problems = []
print(f'Problems: HumanEval={len(humaneval_problems)} MATH-500={len(math500_problems)} Omni-MATH={len(omnimath_problems)}')

In [ ]:
import re, subprocess, sys

def extract_code(text):
    m = re.search(r'''```python\s*\n(.+?)```''', text, re.DOTALL)
    if m: return m.group(1)
    m = re.search(r'''```\s*\n(.+?)```''', text, re.DOTALL)
    if m: return m.group(1)
    return text

def eval_humaneval(code, problem, timeout=5.0):
    script = f'{code}\n\n{problem["test"]}\n\ncheck({problem["entry_point"]})\nprint("PASS")'
    try:
        r = subprocess.run([sys.executable, '-c', script], capture_output=True, text=True, timeout=timeout)
        return {'passed': r.returncode == 0 and 'PASS' in r.stdout,
                'error': r.stderr.strip()[:200] if r.returncode != 0 else None}
    except subprocess.TimeoutExpired:
        return {'passed': False, 'error': f'Timeout {timeout}s'}

def normalize_math(s):
    if not s: return ''
    m = re.search(r'''\\boxed\{(.+?)\}''', s, re.DOTALL)
    if m: s = m.group(1)
    else:
        m = re.search(r'[Aa]nswer\s*:\s*(.+?)(?:\n|$)', s)
        if m: s = m.group(1)
    s = re.sub(r'''\\frac\{([^}]+)\}\{([^}]+)\}''', r'\1/\2', s)
    s = s.strip().strip('$').strip().rstrip('.')
    return re.sub(r'\s+', '', s).lower()

def eval_math(output, ref):
    mn, rn = normalize_math(output), normalize_math(ref)
    passed = mn == rn
    if not passed:
        nums_m = re.findall(r'-?\d+(?:[./]\d+)?', mn)
        nums_r = re.findall(r'-?\d+(?:[./]\d+)?', rn)
        if nums_m and nums_r and nums_m[-1] == nums_r[-1]:
            passed = True
    return {'passed': passed, 'model_answer': mn, 'ref_answer': rn}

print('Eval utilities ready')

In [ ]:
# ── Cell 4: Load results + score ──────────────────────────────────────────────
import json, re, subprocess, sys
from pathlib import Path

all_run_results = {}
for ds_name in ['humaneval', 'math500', 'omnimath']:
    p = RESULTS_DIR / f'{ds_name}_results.json'
    if p.exists():
        with open(p) as f: all_run_results[ds_name] = json.load(f)
        print(f'  {ds_name:12s}: {len(all_run_results[ds_name])} results')
    else:
        print(f'  {ds_name:12s}: not found — skipping')

prob_maps = {
    'humaneval': {p['id']: p for p in humaneval_problems},
    'math500':   {p['id']: p for p in math500_problems},
    'omnimath':  {p['id']: p for p in omnimath_problems},
}

def score_dataset(results, ds_name):
    prob_map = prob_maps[ds_name]
    passed = errors = 0
    for r in results:
        if 'error' in r: errors += 1; continue
        prob = prob_map.get(r['id'])
        if not prob: continue
        if ds_name == 'humaneval':
            ok = eval_humaneval(extract_code(r.get('final_answer', '')), prob)['passed']
        else:
            ok = eval_math(r.get('final_answer', ''), prob.get('answer', ''))['passed']
        if ok: passed += 1
    total = len(results)
    return {'dataset': ds_name, 'total': total, 'passed': passed,
            'errors': errors, 'accuracy': passed/total*100 if total else 0}

scores = {}
for ds_name, results in all_run_results.items():
    s = score_dataset(results, ds_name)
    scores[ds_name] = s
    print(f'{ds_name:18s}: {s["passed"]}/{s["total"]} ({s["accuracy"]:.1f}%)  errors={s["errors"]}')

In [ ]:
import matplotlib.pyplot as plt, numpy as np

paper_data = {
    'HumanEval': {'single_agent': 18.29, 'random_topo': 80.49, 'agentscope': 80.00, 'dytopo_paper': 89.63},
    'APPS-Comp': {'single_agent':  9.62, 'random_topo': 24.24, 'agentscope': 23.12, 'dytopo_paper': 25.14},
    'MATH-500':  {'single_agent': 48.57, 'random_topo': 68.57, 'agentscope': 72.86, 'dytopo_paper': 75.71},
    'Omni-MATH': {'single_agent': 21.43, 'random_topo': 25.71, 'agentscope': 35.71, 'dytopo_paper': 51.43},
}
ds_labels = list(paper_data.keys())
ds_map    = {'HumanEval': 'humaneval', 'APPS-Comp': 'apps', 'MATH-500': 'math500', 'Omni-MATH': 'omnimath'}
x = np.arange(len(ds_labels)); width = 0.18
COLORS = {'single_agent':'#475569', 'random_topo':'#f87171', 'agentscope':'#fbbf24',
          'dytopo_paper':'#4ade80', 'dytopo_local':'#38bdf8'}

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.patch.set_facecolor('#0a0e1a')

ax = axes[0]; ax.set_facecolor('#0f1828')
for i, m in enumerate(['single_agent','random_topo','agentscope','dytopo_paper']):
    ax.bar(x+(i-1.5)*width, [paper_data[d][m] for d in ds_labels], width,
           label=m.replace('_',' ').title(), color=COLORS[m], alpha=0.85)
local_vals = [scores.get(ds_map[d], {}).get('accuracy') for d in ds_labels]
if any(v is not None for v in local_vals):
    ax.scatter(x, [v or 0 for v in local_vals], marker='*', s=300,
               color=COLORS['dytopo_local'], zorder=5, label=f'DyTopo local ({MODEL_LABEL})')
ax.set_xticks(x); ax.set_xticklabels(ds_labels, color='white', fontsize=9)
ax.set_ylabel('Accuracy (%)', color='white')
ax.set_title('DyTopo vs Baselines', color='white')
ax.legend(facecolor='#1a2235', labelcolor='white', fontsize=8)
ax.tick_params(colors='white'); ax.spines[:].set_color('#1e293b'); ax.grid(axis='y', alpha=0.2)

ax2 = axes[1]; ax2.set_facecolor('#0f1828')
for i, m in enumerate(['random_topo','agentscope','dytopo_paper']):
    ax2.bar(x+(i-1)*width, [paper_data[d][m]-paper_data[d]['single_agent'] for d in ds_labels],
            width, label=m.replace('_',' ').title(), color=COLORS[m], alpha=0.85)
if any(v is not None for v in local_vals):
    ax2.scatter(x, [(scores.get(ds_map[d],{}).get('accuracy') or 0)-paper_data[d]['single_agent']
                    for d in ds_labels], marker='*', s=300, color=COLORS['dytopo_local'],
                zorder=5, label=f'DyTopo local ({MODEL_LABEL})')
ax2.axhline(0, color='#94a3b8', linewidth=1, linestyle='--')
ax2.set_xticks(x); ax2.set_xticklabels(ds_labels, color='white', fontsize=9)
ax2.set_ylabel('Improvement over Single Agent (pp)', color='white')
ax2.set_title('Gain from Multi-Agent Routing', color='white')
ax2.legend(facecolor='#1a2235', labelcolor='white', fontsize=8)
ax2.tick_params(colors='white'); ax2.spines[:].set_color('#1e293b'); ax2.grid(axis='y', alpha=0.2)

plt.tight_layout()
plt.savefig(str(RESULTS_DIR / 'results_comparison.png'), dpi=150, bbox_inches='tight', facecolor='#0a0e1a')
plt.show()
print(f'Saved to {RESULTS_DIR}/results_comparison.png')

In [ ]:
import pandas as pd, matplotlib.pyplot as plt, numpy as np, json
from pathlib import Path

PRICING = {
    'claude-haiku-4-5-20251001': {'in': 0.80,  'out': 4.00},
    'claude-sonnet-4-5':         {'in': 3.00,  'out': 15.00},
    'claude-opus-5':             {'in': 15.00, 'out': 75.00},
    'gpt-4o':                    {'in': 2.50,  'out': 10.00},
    'gpt-4o-mini':               {'in': 0.15,  'out': 0.60},
    'dytopo-model':              {'in': 0.00,  'out': 0.00},
}

def compute_cost(tok_in, tok_out, model):
    p = PRICING.get(model, {'in': 0.0, 'out': 0.0})
    return (tok_in * p['in'] + tok_out * p['out']) / 1_000_000

rows = []
for ds_name, results in all_run_results.items():
    ok = [r for r in results if 'error' not in r]
    if not ok: continue
    ti = sum(r.get('tokens_in',  0) for r in ok)
    to = sum(r.get('tokens_out', 0) for r in ok)
    tt = sum(r.get('tokens_total', ti+to) for r in ok)
    model = ok[0].get('model', LLM_MODEL)
    rows.append({
        'dataset': ds_name, 'problems': len(ok),
        'tokens_in': ti, 'tokens_out': to, 'tokens_total': tt,
        'avg_tok/problem': round(tt/len(ok)),
        'avg_rounds': round(sum(r.get('n_rounds',0) for r in ok)/len(ok), 2),
        'model': model,
        'est_cost_usd': round(compute_cost(ti, to, model), 2),
    })

df = pd.DataFrame(rows)
totals = {'dataset': 'TOTAL', 'problems': df['problems'].sum(),
          'tokens_in': df['tokens_in'].sum(), 'tokens_out': df['tokens_out'].sum(),
          'tokens_total': df['tokens_total'].sum(),
          'avg_tok/problem': round(df['tokens_total'].sum()/max(df['problems'].sum(),1)),
          'avg_rounds': round(df['avg_rounds'].mean(), 2),
          'model': LLM_MODEL, 'est_cost_usd': round(df['est_cost_usd'].sum(), 2)}
df_disp = pd.concat([df, pd.DataFrame([totals])], ignore_index=True)

print(f'\n{"="*70}')
print(f'  Token & Cost  |  model: {LLM_MODEL}  |  backend: {BACKEND}')
print(f'{"="*70}')
print(df_disp[['dataset','problems','tokens_in','tokens_out','tokens_total',
               'avg_tok/problem','avg_rounds','est_cost_usd']].to_string(index=False))
print(f'\n  paper Table 8 reference: HumanEval DyTopo = 9,453 tok/problem')

he = df[df['dataset']=='humaneval']['avg_tok/problem'].values
if len(he):
    print(f'  HumanEval ours={he[0]:,}  paper=9,453  ratio={he[0]/9453:.2f}x')

# ── Charts ────────────────────────────────────────────────────────────────────
ds_names = df['dataset'].tolist()
x = np.arange(len(ds_names)); w = 0.35
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('#0a0e1a')

ax1 = axes[0]; ax1.set_facecolor('#0f1828')
ax1.bar(x-w/2, df['tokens_in']/1e6,  w, label='Prompt (M)',     color='#38bdf8', alpha=0.85)
ax1.bar(x+w/2, df['tokens_out']/1e6, w, label='Completion (M)', color='#f87171', alpha=0.85)
ax1.set_xticks(x); ax1.set_xticklabels(ds_names, color='white')
ax1.set_ylabel('Tokens (millions)', color='white')
ax1.set_title('Token consumption by dataset', color='white')
ax1.legend(facecolor='#1a2235', labelcolor='white', fontsize=9)
ax1.tick_params(colors='white'); ax1.spines[:].set_color('#1e293b'); ax1.grid(axis='y', alpha=0.2)

ax2 = axes[1]; ax2.set_facecolor('#0f1828')
ax2.bar(x, df['avg_tok/problem'], color='#4ade80', alpha=0.85)
if 'humaneval' in ds_names:
    ax2.axhline(9453, color='#fbbf24', linewidth=1.5, linestyle='--', label='Paper Table 8 (9,453)')
ax2.set_xticks(x); ax2.set_xticklabels(ds_names, color='white')
ax2.set_ylabel('Avg tokens / problem', color='white')
ax2.set_title('Token efficiency per problem', color='white')
ax2.legend(facecolor='#1a2235', labelcolor='white', fontsize=9)
ax2.tick_params(colors='white'); ax2.spines[:].set_color('#1e293b'); ax2.grid(axis='y', alpha=0.2)

plt.tight_layout()
plt.savefig(str(RESULTS_DIR / 'token_consumption.png'), dpi=150, bbox_inches='tight', facecolor='#0a0e1a')
plt.show()
print(f'Saved to {RESULTS_DIR}/token_consumption.png')

In [ ]:
# ── Cell 7: Topology edge density evolution ───────────────────────────────────
import matplotlib.pyplot as plt

def compute_density(adj, n):
    if n < 2: return 0.0
    return sum(adj[j][i] for j in range(n) for i in range(n) if j != i) / (n*(n-1))

round_densities = {}
for ds_name, results in all_run_results.items():
    by_round = {}
    for r in results:
        for ridx, adj in enumerate(r.get('adjacency_per_round', [])):
            n = len(adj)
            by_round.setdefault(ridx, []).append(compute_density(adj, n))
    round_densities[ds_name] = {k: sum(v)/len(v) for k, v in by_round.items()}

COLORS_DS = {'humaneval':'#38bdf8','math500':'#fbbf24','omnimath':'#a78bfa'}
fig, ax = plt.subplots(figsize=(9, 5))
ax.set_facecolor('#0f1828'); fig.patch.set_facecolor('#0a0e1a')
for ds_name, by_round in round_densities.items():
    if not by_round: continue
    rounds_x = sorted(by_round.keys())
    ax.plot([r+1 for r in rounds_x], [by_round[r] for r in rounds_x], 'o-',
            color=COLORS_DS.get(ds_name,'#94a3b8'), linewidth=2.5, markersize=8, label=ds_name)
ax.set_xlabel('Round', color='white', fontsize=11)
ax.set_ylabel('Avg Edge Density', color='white', fontsize=11)
ax.set_title('Topology Evolution — dense exploration → sparse finalization (paper Fig 4)',
             color='white', fontsize=10)
ax.legend(facecolor='#1a2235', labelcolor='white', fontsize=9)
ax.tick_params(colors='white'); ax.spines[:].set_color('#1e293b'); ax.grid(alpha=0.2); ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.savefig(str(RESULTS_DIR / 'density_evolution.png'), dpi=150, bbox_inches='tight', facecolor='#0a0e1a')
plt.show()

In [ ]:
# ── Cell 8: Full results summary table ────────────────────────────────────────
paper_dytopo = {'humaneval': 89.63, 'apps': 25.14, 'math500': 75.71, 'omnimath': 51.43}
paper_single = {'humaneval': 18.29, 'apps':  9.62, 'math500': 48.57, 'omnimath': 21.43}

print(f'Model: {MODEL_LABEL}')
print(f'{"─"*65}')
print(f'{"Dataset":18s} {"Ours":>9s} {"Paper DyTopo":>13s} {"Paper SA":>9s} {"Δ vs SA":>10s}')
print(f'{"─"*65}')
for ds in ['humaneval', 'math500', 'omnimath']:
    s = scores.get(ds)
    if not s: continue
    diff = s['accuracy'] - paper_single.get(ds, 0)
    imp  = f'+{diff:.1f}pp' if diff >= 0 else f'{diff:.1f}pp'
    print(f'{ds:18s} {s["accuracy"]:>8.1f}% {paper_dytopo.get(ds,0):>12.1f}% {paper_single.get(ds,0):>8.1f}% {imp:>10s}')
print(f'{"─"*65}')
print(f'All results in: {RESULTS_DIR}')